# 보너스: Adam, Newton, 정보 이론

필수 구현을 수정하지 않고 [Adam/Newton](../src/bonus_optimizer.py)과 [정보 이론](../src/bonus_probability.py)을 상속 모듈로 분리했다. 최적화는 같은 초기점 (5,5), 타원 함수 $f=x^2+10y^2$를 사용했다.

In [1]:
from pathlib import Path
import os
import sys
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.environ['MPLCONFIGDIR'] = str(ROOT / '.cache' / 'matplotlib')
import numpy as np
np.random.seed(42)
print('Python:', sys.executable)
print('NumPy:', np.__version__)
import matplotlib.pyplot as plt
plt.rcParams["axes.prop_cycle"] = plt.cycler(color=["#14213d","#a67c20","#64748b","#1d4ed8"])
from src.optimizer import VanillaGD, Momentum, optimize, ellipse_loss, ellipse_gradient, plot_paths, plot_loss_curves
from src.bonus_optimizer import Adam, NewtonMethod
from src.bonus_probability import InformationTheory
from scripts.run_experiments import convergence_summary


Python: /Users/oliverjoo/Dev/Anaconda/anaconda3/envs/py312/bin/python
NumPy: 1.26.4


## 1. Adam

$$m_t=\beta_1m_{t-1}+(1-\beta_1)g_t,\quad v_t=\beta_2v_{t-1}+(1-\beta_2)g_t^2$$
$$\hat m_t=\frac{m_t}{1-\beta_1^t},\quad \hat v_t=\frac{v_t}{1-\beta_2^t},\quad \theta_{t+1}=\theta_t-\eta\frac{\hat m_t}{\sqrt{\hat v_t}+\epsilon}$$

초기 모멘트가 0이어서 작게 추정되는 현상을 편향 보정으로 보완했다. GD/Momentum/Adam 모두 lr=.01, Adam β1=.9/β2=.999/ε=1e-8을 사용했다. 그래프는 2000회까지, 경로는 첫 200회만 표시했다. 손실 .01 이하 **최초 도달**을 속도 기준으로 삼았다. 최초 도달은 그 뒤에도 계속 그 이하라는 뜻은 아니다.

Adam은 좌표별로 업데이트 크기를 조절하지만 이 간단한 이차 함수에서 항상 가장 빠르다는 보장은 없다.

In [2]:
paths = {name:optimize(optimizer,ellipse_gradient,steps=2000) for name,optimizer in
         [('GD lr=0.01',VanillaGD(.01)),('Momentum lr=0.01',Momentum(.01,.9)),('Adam lr=0.01',Adam(.01))]}
figure=plot_paths(ellipse_loss,{k:v[:201] for k,v in paths.items()},'Bonus: first 200 updates, same lr=0.01')
figure.savefig(ROOT/'outputs'/'bonus_adam_paths.png',dpi=150,bbox_inches='tight')
plt.show()
figure=plot_loss_curves(ellipse_loss,paths,'Bonus: GD / Momentum / Adam, same lr=0.01')
figure.savefig(ROOT/'outputs'/'bonus_adam_loss.png',dpi=150,bbox_inches='tight')
plt.show()
convergence_summary(paths,ellipse_loss)

/Users/oliverjoo/Dev/codyssey/2026/codyssey_missions/AI_SW/02_Advanced/A2-1/.cache/tmp/ipykernel_59822/3408324640.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/Users/oliverjoo/Dev/codyssey/2026/codyssey_missions/AI_SW/02_Advanced/A2-1/.cache/tmp/ipykernel_59822/3408324640.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


{'GD lr=0.01': {'final_point': [1.416189562727233e-17, 7.567352911521175e-194],
  'final_loss': 2.005592877577551e-34,
  'final_radius': 1.416189562727233e-17,
  'first_update_loss_le_0.01': 194,
  'updates': 2000},
 'Momentum lr=0.01': {'final_point': [8.672220178042903e-46,
   -8.794988643572526e-46],
  'final_loss': 8.487256552221516e-90,
  'final_radius': 1.2351486876365298e-45,
  'first_update_loss_le_0.01': 78,
  'updates': 2000},
 'Adam lr=0.01': {'final_point': [3.263542196129379e-05,
   3.263542042481277e-05],
  'final_loss': 1.1715777429634562e-08,
  'final_radius': 4.6153455264974296e-05,
  'first_update_loss_le_0.01': 1242,
  'updates': 2000}}

## 2. Newton Method

$$H=\begin{bmatrix}2&0\\0&20\end{bmatrix},\quad g=\begin{bmatrix}2x\\20y\end{bmatrix},\quad \theta_{t+1}=\theta_t-H^{-1}g$$

역행렬을 만들지 않고 `np.linalg.solve(H,g)`로 $H\Delta=g$를 풀었다. 이 함수는 Hessian이 일정하고 양의 정부호이며 $H^{-1}g=\theta$이므로 한 번에 원점에 도달한다. 일반 비선형 함수에서도 한 번에 끝난다는 뜻은 아니다. 변수 수 d에 대해 Hessian은 d×d 공간이 필요하고 행렬 풀이 비용도 증가한다. 특이하거나 부정정인 Hessian에서는 단순 Newton 업데이트가 실패하거나 최소점 방향을 보장하지 못한다. 추가 댐핑 알고리즘은 과제 범위에 넣지 않았다.

In [3]:
paths = {'GD lr=0.01':optimize(VanillaGD(.01),ellipse_gradient,steps=100),
         'Newton':optimize(NewtonMethod(lambda point:np.diag([2.,20.])),ellipse_gradient,steps=100)}
figure=plot_paths(ellipse_loss,paths,'Bonus: analytic Hessian Newton vs GD')
figure.savefig(ROOT/'outputs'/'bonus_newton_paths.png',dpi=150,bbox_inches='tight')
plt.show()
figure=plot_loss_curves(ellipse_loss,paths,'Bonus: Newton solves the quadratic in one update')
figure.savefig(ROOT/'outputs'/'bonus_newton_loss.png',dpi=150,bbox_inches='tight')
plt.show()
convergence_summary(paths,ellipse_loss)

/Users/oliverjoo/Dev/codyssey/2026/codyssey_missions/AI_SW/02_Advanced/A2-1/.cache/tmp/ipykernel_59822/3607639793.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/Users/oliverjoo/Dev/codyssey/2026/codyssey_missions/AI_SW/02_Advanced/A2-1/.cache/tmp/ipykernel_59822/3607639793.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


{'GD lr=0.01': {'final_point': [0.6630977794737659, 1.018517988167243e-09],
  'final_loss': 0.43969866514303907,
  'final_radius': 0.6630977794737659,
  'first_update_loss_le_0.01': None,
  'updates': 100},
 'Newton': {'final_point': [0.0, 0.0],
  'final_loss': 0.0,
  'final_radius': 0.0,
  'first_update_loss_le_0.01': 1,
  'updates': 100}}

## 3. 엔트로피, KL-Divergence, Cross-Entropy

$$H(p)=-\sum_jp_j\ln p_j$$
$$D_{KL}(p\Vert q)=\sum_jp_j\ln\frac{p_j}{q_j}$$
$$H(p,q)=-\sum_jp_j\ln q_j=H(p)+D_{KL}(p\Vert q)$$

p는 실제 분포, q는 예측 분포다. 동전이 공정하면 결과를 예측하기 어려워 엔트로피가 높다. 이미 정해진 결과는 엔트로피 0이다. 실제 분포와 다른 예측을 사용할 때 생기는 추가 로그 손실이 KL이다. KL은 대칭 거리가 아니다.

자연로그를 사용해 단위는 **nats**다. $0\ln0=0$으로 처리했다. 실제로 가능한 사건(p>0)에 예측 확률을 0으로 주면 CE/KL은 무한대다.

In [4]:
p,q=np.array([.3,.7]),np.array([.6,.4])
entropy=InformationTheory.entropy(p)
kl=InformationTheory.kl_divergence(p,q)
ce=InformationTheory.cross_entropy(p,q)
print('H(p)=',entropy,'KL(p||q)=',kl,'CE(p,q)=',ce,'unit=nats')
assert abs(ce-entropy-kl)<1e-12
print('identity error:',abs(ce-entropy-kl))
print('certain event entropy:',InformationTheory.entropy([1,0]))

H(p)= 0.6108643020548935 KL(p||q)= 0.18378689738681217 CE(p,q)= 0.7946511994417057 unit=nats
identity error: 2.7755575615628914e-17
certain event entropy: -0.0
